# Atlantis Zarr Datacube + STAC Catalog — a first worked example

> **STAC = "what to load"** (catalog / metadata / query layer)
>
> **Zarr = "how to load efficiently"** (chunked, cloud-native data layer)

This notebook is a guided, **read-only** tour of Atlantis' first real example
datacube — the Valencia 2024 VIIRS flood archive at
`s3://atlantis/zarr/viirs_archive_2024/` — and its companion STAC catalog. It
covers everything described in [`docs/archive/zarr-spec.md`](../docs/archive/zarr-spec.md)
and [`docs/stac_zarr.md`](../docs/stac_zarr.md): the canonical global grid,
chunk/shard/codec/metadata introspection of the live store, the public
`ArchiveReader` API (including ML tile reads), the STAC Collection/Item
structure, the `stac-geoparquet` scale path, and a matplotlib+cartopy per-day
plot (the static-image analogue of `atlantis viz serve --basemap`).

**Audience:** newcomers who want to understand the storage model end to end,
and advanced users who want copy-pasteable read patterns.

**Prerequisites:**
- Run inside the `notebooks` pixi environment: `pixi run -e notebooks jupyter lab` (or `pixi shell -e notebooks`).
- AWS credentials for the ECMWF S3-compatible object store — run `atlantis setup` once, or ensure `~/.aws/config`'s `[default]` profile has `endpoint_url = https://object-store.os-api.cci1.ecmwf.int` (see `src/atlantis/utils/setup.py`).
- Optional, only for the `xpystac` fast-path in §4: `pixi run -e stac ...` or `pip install atlantis[stac]`. Everything else runs with plain `xarray`/`zarr`/`pystac`/`geopandas`.

**Scope:** every cell below only *reads* from `s3://atlantis`. Nothing here
writes to, or mutates, the shared bucket.

In [ ]:
%matplotlib inline

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    """Walk upward from *start* until a directory containing pyproject.toml is found."""
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not locate the atlantis repo root (no pyproject.toml found).")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import numpy as np
import xarray as xr

print(f"Repo root: {REPO_ROOT}")

## 0. Configuration — target store & AOI

Target the real, already-populated example store for this walkthrough
(Valencia 2024, VIIRS) and the same AOI/date range used throughout the docs
and CLI demos.

In [ ]:
from atlantis.config import get_config

# The real, already-populated example store for this walkthrough.
ARCHIVE_ROOT = "s3://atlantis/zarr/viirs_archive_2024"
STAC_ROOT = f"{ARCHIVE_ROOT}/stac"
SOURCE = "viirs"

# Valencia 2024 flood — same AOI/date range used throughout the docs & CLI demos.
BBOX = (-1.5, 38.8, 0.5, 40.0)  # west, south, east, north
START, END = "2024-10-29", "2024-11-04"

config = get_config()
# Empty by default -> None, so s3fs/boto3 fall back to the default AWS profile
# (endpoint_url configured for the ECMWF object store by `atlantis setup`).
storage_options = config.archive.storage_options or None

print(f"Archive root : {ARCHIVE_ROOT}")
print(f"STAC root    : {STAC_ROOT}")
print(f"BBox         : {BBOX}")
print(f"Date range   : {START} -> {END}")
print(f"storage_options (ArchiveConfig): {storage_options!r}")

## 1. The canonical global grid

Every source group in the datacube is co-registered on the same fixed
1-arcmin global grid (`src/atlantis/archive/grid.py`) — the same grid used by
ECMWF `Globe_flood_area_*.grb` / `CMF_all.zarr`, so harmonised outputs stack
with global products out of the box.

In [ ]:
from atlantis.archive import grid

print(f"CRS               : {grid.GLOBAL_CRS}")
print(f"Resolution        : {grid.GLOBAL_RESOLUTION:.6f}\u00b0 (1 arc-minute)")
print(f"Shape (rows, cols): {grid.GLOBAL_HEIGHT} x {grid.GLOBAL_WIDTH}")
print(f"Origin            : lon={grid.ORIGIN_LON}, lat={grid.ORIGIN_LAT} (pixel-centre convention)")

window = grid.bounds_to_window(*BBOX)
print(f"\nValencia bbox {BBOX}")
print(f"  -> IndexWindow(row={window.row_start}:{window.row_stop}, col={window.col_start}:{window.col_stop})")
print(f"  -> window size: {window.height} x {window.width} px")

## 2. Live introspection of the real Zarr store

Open the raw store directly with the `zarr` library (no `xarray` yet) to see
exactly what is on S3 — this should match:

```text
aws s3 ls s3://atlantis/zarr/viirs_archive_2024/datacube.zarr/viirs/
                           PRE crs/
                           PRE flood_fraction/
                           PRE time/
                           PRE x/
                           PRE y/
2026-07-07 16:06:11       8366 zarr.json
```

In [ ]:
import zarr

from atlantis.archive._store import store_for

store = store_for(ARCHIVE_ROOT, config.archive.store, storage_options)
root = zarr.open_group(store, mode="r")

sources = sorted(root.group_keys())
print(f"Sources (groups) in the datacube: {sources}")

viirs_group = root[SOURCE]
print(f"\nArrays in '{SOURCE}' group: {sorted(viirs_group.array_keys())}")

In [ ]:
flood_fraction = viirs_group["flood_fraction"]

print("flood_fraction array")
print(f"  shape      : {flood_fraction.shape}   (time, y, x)")
print(f"  dtype      : {flood_fraction.dtype}")
print(f"  chunks     : {flood_fraction.chunks}   <- data-loader read unit (inner chunk)")
print(f"  shards     : {getattr(flood_fraction, 'shards', None)}   <- cloud storage-object unit")
print(f"  fill_value : {flood_fraction.fill_value}")
try:
    print(f"  codecs     : {flood_fraction.codecs}")
except AttributeError:
    pass
print(f"  attrs      : {dict(flood_fraction.attrs)}")

print("\nGroup-level provenance attrs:")
group_attrs = dict(viirs_group.attrs)
for key in ("source_id", "last_updated", "crs"):
    print(f"  {key}: {group_attrs.get(key)}")
print(f"  atlantis_events: {list(group_attrs.get('atlantis_events', {}).keys())}")

time_arr = viirs_group["time"]
print(f"\ntime coordinate: {time_arr.shape[0]} date(s)")
print(f"  attrs: {dict(time_arr.attrs)}")
if time_arr.shape[0]:
    print(f"  first/last raw int values: {time_arr[0]} / {time_arr[-1]}")

## 3. Reading via the public API — `ArchiveReader`

Most users should not touch zarr/xarray internals directly —
`ArchiveReader.read()` (`src/atlantis/archive/reader.py`) handles bbox→window
mapping, CF-decoding, and time-slicing for you.

In [ ]:
from atlantis.archive.reader import ArchiveReader

reader = ArchiveReader(ARCHIVE_ROOT, config.archive, storage_options=storage_options)

print(f"Sources : {reader.list_sources()}")
print(f"Events  : {reader.list_events()}")

ds = reader.read(SOURCE, bbox=BBOX, start=START, end=END)
print(f"\nWindowed dataset — dims: {dict(ds.sizes)}")
print(f"Variables: {list(ds.data_vars)}")
print(f"flood_fraction dtype: {ds['flood_fraction'].dtype}  <- CF-decoded to float+NaN (uint8 [0,100] on disk)")

> **Caveat** (`docs/archive/zarr-spec.md` §8): the default `open_zarr`/
> `ArchiveReader.read()` call uses CF decoding, so *every* variable —
> including the categorical masks — comes back as `float64` with `NaN` for
> the fill value, not `uint8`. Open with `mask_and_scale=False` (e.g.
> `xr.open_zarr(..., mask_and_scale=False)`) if you need the raw integer
> codes for `quality_mask`/`permanent_water`.

### Tile-based reads for ML data loaders

Pass `tiles=[(row, col), ...]` to get fixed `chunk_size x chunk_size`
(256x256) blocks stacked on a new `tile` dimension — the intended batch unit
for a PyTorch `Dataset`/`DataLoader` (edge tiles are NaN-padded). These
`(row, col)` indices are **absolute positions in the global 10800x21600
grid** — combine with `bbox` only after translating them into the windowed
subset's local index space.

In [ ]:
# Pick a couple of absolute tile indices intersecting the Valencia window.
tile_size = config.archive.chunk_size
row0, col0 = window.row_start // tile_size, window.col_start // tile_size
tiles = [(row0, col0), (row0, col0 + 1)]
print(f"Requesting tiles {tiles} ({tile_size}x{tile_size} px each, no bbox/time filter)")

ds_tiles = reader.read(SOURCE, tiles=tiles)
print(f"Tiled dataset dims: {dict(ds_tiles.sizes)}")
print(f"tile_row: {ds_tiles.coords['tile_row'].values}, tile_col: {ds_tiles.coords['tile_col'].values}")

## 4. STAC discovery — the static catalog

STAC answers *"what to load"*; Zarr answers *"how to load it efficiently"*
(`docs/stac_zarr.md`). The example bucket also carries a pre-built static
catalog:

```text
aws s3 ls s3://atlantis/zarr/viirs_archive_2024/stac/
                           PRE atlantis-datacube-viirs/
2026-07-07 22:17:33        582 catalog.json
2026-07-08 09:09:19      37002 items.parquet
```

This section only **reads** that pre-built catalog — it never rebuilds or
overwrites it (rebuilding via `build_datacube_catalog()`/`write_catalog()`
would re-scan the whole remote cube and is left to the CLI's
`atlantis stac build`, see `docs/stac_zarr.md` §7 for the data-flow diagram).

In [ ]:
import pystac

from atlantis.stac import FsspecStacIO

catalog_href = f"{STAC_ROOT}/catalog.json"
catalog = pystac.Catalog.from_file(catalog_href, stac_io=FsspecStacIO(storage_options))

print(f"Catalog id    : {catalog.id}")
print(f"Catalog title : {catalog.title}")

collections = list(catalog.get_children())
print(f"Collections   : {[c.id for c in collections]}")

viirs_collection = next(c for c in collections if c.id.endswith(SOURCE))
print(f"\nUsing collection: {viirs_collection.id}")
print(f"cube:dimensions: {viirs_collection.extra_fields.get('cube:dimensions')}")
print(f"cube:variables : {list(viirs_collection.extra_fields.get('cube:variables', {}))}")

zarr_asset = viirs_collection.assets["zarr"]
print(f"\nzarr asset href       : {zarr_asset.href}")
print(f"zarr asset open_kwargs: {zarr_asset.extra_fields.get('xarray:open_kwargs')}")

In [ ]:
import itertools

# Counting via item *links* is cheap (already parsed from collection.json);
# fetching each Item's own JSON (get_items()) is a separate S3 GET per item,
# so we only preview a handful rather than fetching all 366.
item_links = viirs_collection.get_links("item")
print(f"{len(item_links)} item(s) (one per populated date) linked in this collection\n")

preview = list(itertools.islice(viirs_collection.get_items(), 5))
print(f"Previewing the first {len(preview)} item(s):")
for item in preview:
    print(f"  {item.id:24s} datetime={item.datetime}  bbox={item.bbox}")

### Opening the STAC `zarr` asset as xarray

Two ways to open the same store the Collection points at:

1. **Manual** (works anywhere — only needs `xarray`/`zarr`): read the asset's
   `xarray:open_kwargs` and call `xr.open_zarr` directly.
2. **Convenience** (needs the optional `xpystac` package, only in the `stac`
   pixi env): `xr.open_dataset(asset, engine="stac")`.

In [ ]:
open_kwargs = dict(zarr_asset.extra_fields["xarray:open_kwargs"])
open_kwargs.pop("engine", None)  # "zarr" - implied by xr.open_zarr(), not an accepted kwarg there
ds_from_stac = xr.open_zarr(zarr_asset.href, **open_kwargs)
ds_from_stac_windowed = ds_from_stac.isel(
    y=slice(window.row_start, window.row_stop),
    x=slice(window.col_start, window.col_stop),
).sel(time=slice(np.datetime64(START), np.datetime64(END)))

print(f"Manually opened via the STAC asset — dims: {dict(ds_from_stac_windowed.sizes)}")
print(f"Matches the ArchiveReader window: {dict(ds_from_stac_windowed.sizes) == dict(ds.sizes)}")

In [ ]:
try:
    import xpystac  # noqa: F401 - registers the "stac" xarray engine

    ds_xpystac = xr.open_dataset(zarr_asset, engine="stac")
    print(f"xpystac fast-path OK — dims: {dict(ds_xpystac.sizes)}")
except ImportError:
    print(
        "xpystac not installed — skipping the engine='stac' fast path.\n"
        "Install it with `pixi run -e stac ...` or `pip install atlantis[stac]` for this shortcut;\n"
        "the manual xr.open_zarr() approach above always works without it."
    )

try:
    from atlantis.viz.dashboard import from_stac

    ds_helper = from_stac(STAC_ROOT, SOURCE, bbox=BBOX, start=START, end=END, storage_options=storage_options)
    print(f"\natlantis.viz.dashboard.from_stac() OK — dims: {dict(ds_helper.sizes)}")
except ImportError as exc:
    print(f"\natlantis.viz.dashboard.from_stac() needs the 'stac' extra (xpystac): {exc}")

## 5. The scale path — stac-geoparquet

For thousands of items, walking the JSON tree gets slow. `items.parquet` is a
single columnar index of every item, queryable with plain `geopandas` — **no
`pystac`/`xpystac` needed** for querying (only for *writing* it via
`export_items_to_geoparquet()`).

In [ ]:
from atlantis.stac.geoparquet import search_geoparquet

items_parquet = f"{STAC_ROOT}/items.parquet"
gdf = search_geoparquet(items_parquet, bbox=BBOX, start=START, end=END)

print(f"{len(gdf)} matching row(s) in {items_parquet}")
print(f"Columns: {list(gdf.columns)}")
gdf[["id", "datetime"]].head(10) if "id" in gdf.columns else gdf.head(10)

## 6. Putting it together — per-day plot (matplotlib + cartopy basemap)

The static-image analogue of `atlantis viz serve --basemap`
(`src/atlantis/cli.py`) and `scripts/datacube_walkthrough.py`: loop over the
populated Valencia dates and render each with a coastline/border overlay.

In [ ]:
import matplotlib.pyplot as plt

from atlantis.utils.plot import pixel_stats_classified

try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature

    HAS_CARTOPY = True
except ImportError:
    HAS_CARTOPY = False
    print("cartopy not installed — plotting without a basemap overlay.")


def plot_day(da, title):
    """Render one day of flood_fraction, with a coastline/border basemap if available."""
    vmax = max(float(np.nanmax(da.values)), 0.01)

    # Match the figure aspect ratio to the AOI's lon/lat extent so cartopy's
    # equal-aspect projection doesn't letterbox the axes — which otherwise makes
    # the auto colorbar taller than the actual rendered map ("overflowing").
    lon_span = float(da["x"].max() - da["x"].min()) or 1.0
    lat_span = float(da["y"].max() - da["y"].min()) or 1.0
    height = 6.0
    width = float(np.clip(height * lon_span / lat_span, 4.0, 12.0))

    fig = plt.figure(figsize=(width, height))
    cbar_kwargs = {"label": "flood_fraction", "shrink": 0.9, "pad": 0.05}
    if HAS_CARTOPY:
        ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
        da.plot(
            ax=ax,
            transform=ccrs.PlateCarree(),
            cmap="Blues",
            vmin=0.0,
            vmax=vmax,
            add_colorbar=True,
            cbar_kwargs=cbar_kwargs,
        )
        ax.add_feature(cfeature.COASTLINE.with_scale("10m"), linewidth=0.6)
        ax.add_feature(cfeature.BORDERS.with_scale("10m"), linewidth=0.4, linestyle=":")
        # Only label the bottom/left axes (not mirrored on top/right). Note:
        # passing the sides directly here (rather than setting
        # gl.top_labels/gl.right_labels after the fact) avoids a cartopy
        # tight-bbox rendering bug that can blank out the map entirely.
        ax.gridlines(draw_labels=["bottom", "left"], linewidth=0.3, alpha=0.5)
    else:
        ax = fig.add_subplot(1, 1, 1)
        da.plot(ax=ax, cmap="Blues", vmin=0.0, vmax=vmax, add_colorbar=True, cbar_kwargs=cbar_kwargs)
        ax.set_xlabel("Longitude (°)")
        ax.set_ylabel("Latitude (°)")
    ax.set_title(title, fontsize=10, fontweight="bold")
    plt.show()
    plt.close(fig)

In [ ]:
da_full = ds["flood_fraction"]
for t in da_full["time"].values:
    day_label = np.datetime_as_string(t, unit="D")
    da_day = da_full.sel(time=t)
    if bool(da_day.isnull().all()):
        print(f"{day_label}: no data - skipping")
        continue
    pixel_stats_classified(da_day.values, name=day_label)
    plot_day(da_day, f"Valencia 2024: VIIRS flood_fraction {day_label}")

## 7. Recap & where to go next

**Points for ML review** (`docs/archive/zarr-spec.md` §8):
- Tile size `256x256`, dtype `uint8` on disk (decodes to float `[0,1]`) —
  confirm this matches model input expectations (patch size, `/32`
  divisibility for U-Net encoders).
- Normalisation: `flood_fraction` decodes to physical `[0,1]` via CF
  `scale_factor` — no additional per-image rescale is applied. Masks decode
  to `float64`+`NaN` by default; use `mask_and_scale=False` for the raw
  categorical `uint8` codes.
- NODATA=255 → `NaN` after CF decode; confirm the masking/ignore-index
  convention expected by your loss function.
- Random access: `256²` chunks packed into `2048²` shards — fine-grained tile
  reads vs. a few large cloud objects.
- Channels: `flood_fraction`, `quality_mask`, `permanent_water` (+
  `recurring_flood` for MODIS).
- ML loader stubs (`validation/ml_loader.py`) are not yet implemented.

**Further reading:**
- `docs/archive/zarr-spec.md` — full datacube schema
- `docs/stac_zarr.md` — STAC + visualization layer
- `scripts/datacube_walkthrough.py` — CLI-runnable equivalent of §6 (writes PNGs to disk)
- `atlantis stac build` / `atlantis viz serve` — CLI commands over this same datacube
- `tests/archive/`, `tests/stac/` — further usage examples

**This notebook is read-only** — it never calls `ArchiveWriter.write()`,
`write_catalog()`, or `export_items_to_geoparquet()` against the shared
`s3://atlantis` bucket.